# Simulate Anaerobic Conditions in the S. Tm Context GEMs

## Background
I am stuck with models that predict the same growth rates in `meeting_052126.ipynb`. It seems to be an issue with how and when the models are saved. FBA simulates different growth rates when a model is generated from the gene weights step and simulations are run in the same file. I will regenerate models from the gene weights step in each notebook until I find the cause of this.  

## Objective
- Find "where" the model can be adjusted for oxygen levels
- Reduce all oxygen to simulate anaerobic environment
- Run FBA

### 1. Convert from gene weights to reaction weights 

In [1]:
import os
from cobra.core.configuration import Configuration

# Set the Gurobi license and solver
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [2]:
from cobra.io import load_model, load_json_model

# load the test S.Tm LT2 GEM in cobrapy
base_model = load_model("STM_v1_0")

/Users/gina/miniforge3/envs/localgems/lib/python3.11/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

GurobiError: Unable to open Gurobi license file '/home/gmvaz/.gurobi.lic'

In [ ]:
import pandas as pd

# read in the gene weights csv files of the different threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")

In [ ]:
# make each list of gene weights into a python set
stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

# check the length
print(len(stm_set_10_90))
print(len(stm_set_15_85))
print(len(stm_set_25_75))

In [ ]:
# make a list of all genes in the base model and save it as a set
model_genes = set([g.id for g in base_model.genes])

#check
print(len(model_genes))

In [ ]:
# subset the gene weights list for each model/sample to only have the genes present in the input model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

In [ ]:
# there are different amounts of genes in the gene weights lists and in the input model gene list. 
# make a new gene weight list per sample that has all the genes from the input model:
# (1) assign a value of 0 to the 20 genes found in the input model but not in the gene weights list. 
# (2) add the genes that are present in the input model and raw gene weight list. 
# Result: a gene weight list per sample that has only the genes found in the input model and raw gene weights list. 
subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

# check
print(len(subset_weights_stm1090))
print(len(subset_weights_stm1585))
print(len(subset_weights_stm2575))

### 2. Convert gene weights to reaction weights

In [ ]:
from imatpy.parse_gpr import gene_to_rxn_weights

# convert gene weights to reaction weights using the final gene weights lists with all genes from the input model
rxn_weights_stm1090 = gene_to_rxn_weights(base_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(base_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(base_model, subset_weights_stm2575)

# check
print(len(rxn_weights_stm1090))
print(len(rxn_weights_stm1585))
print(len(rxn_weights_stm2575))

### 3. Generate context specific models with imat

In [ ]:
from imatpy.model_creation import generate_model

# generate context-specific models with imat by integrating the reaction weights
stm1090_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

### Run FBA

In [ ]:
# run FBA on all models

fba_stm1090 = stm1090_model.optimize()
fba_stm1585 = stm1585_model.optimize()
fba_stm2575 = stm2575_model.optimize()
fba_base = base_model.optimize()

In [ ]:
# print the objective value for each model after FBA

print(fba_base.objective_value)
print(fba_stm1090.objective_value)
print(fba_stm1585.objective_value)
print(fba_stm2575.objective_value)

### 4. Onto the new stuff - Where's the oxygen??

Taking a break is always good. Last Thursday, 05/21, I was going crazy looking through the Cobra docs to figure out where to change the environmental constraints. Turns out, in papers and the cobra textbook, some people say 'environmental constraints' and mean nutrient availability in the model but other times they like to spruce things up and call it 'medium' instead. Medium - as in the growth media of nutrients available to the model... I don't know why I didn't read this sooner in the docs. I guess confusion, too many words for the same thing, etc etc. 

Anyways, let's go through the [demo](https://cobrapy.readthedocs.io/en/latest/media.html) for the growth media `medium` attribute of the model. 

In [ ]:
base_model.medium

The growth media of a model is managed by the `medium` attribute. `model.medium` returns a dictionary that has the upper bounds of all the non-zero exchange fluxes (the active exchange reactions). The growth media of a model can be modified by assigning a dictionary to `model.medium` that define exchange reactions with new upper import bounds. Note, you cannot assign directly to the `medium` attribute of a model; this won't work: 
```
model.medium["EX_co2_e"] = 0.0
```
The output for `EX_co2_e` will stay the same in this case. 

First, let's enforce anaerobic growth by shutting off oxygen import. 

In [ ]:
no_oxygen_medium = base_model.medium

no_oxygen_medium["EX_o2_e"] = 0.0
new_model = base_model.copy()

new_model.medium = no_oxygen_medium
new_model.medium

Setting the upper bound of oxygen import to 0 removes the exchange reaction from the list of active exchange. 

Next, let's run FBA for the `base_model` to see the difference in growth rate in an anaerobic environment. 

In [ ]:
fba_base_anaerobic = new_model.optimize()
print(fba_base_anaerobic.objective_value)

### 05/28/26 Meeting with Titus
**Questions:** 
- If I refresh the browser, are the variables still there? What if I leave Jupyter running on a tmux session but close the ssh tunnel thing?
- How does it work (with the variable name) when you run FBA on a model but then change the conditions and run it again on the same model?
- why do something have () and other don't?
- how to look at attributes?

Use the `assert` function to check the value of something is what you expect it to be. 

In [ ]:
# when making copies of a model, it's best to reload it to prevent copying an already-altered model
play_model = load_model("STM_v1_0")

# we expect 18.5 to be the upper bound for the oxygen exchange reaction
# if it's true then assert won't print an output, if false it'll print an error
assert play_model.medium["EX_o2_e"] == 18.5

# we want to check if the upper bound of oxygen is 0, 
# if it's true then the function won't print an output. 
# if false, it'll print the actual value. This is denoted by what goes after the comma
assert play_model.medium["EX_o2_e"] == 0, play_model.medium["EX_o2_e"]

Define functions for frequently used code. 

In [ ]:
# we are making a function named `change_medium` that
# takes `model` and `new_medium` as arguments 
# `model` is an object with a `.copy()` method. 
# We copy the model and save it in a new variable named`new_model`
# `dict()` is a function that copies a dictionary. 
# `new_medium` is a dictionary that is being copied
# `new_medium` is assigned to `new_model.medium` (which has the originial medium copied)
# the function returns `new_model` which is a copy of `model` and assigns `new_medium` to the medium attribute

# Questions: 
# Why do we have to copy `new_medium` before assigning it to the model if
# `new_medium` is a dictionary that is made prior to using the function? 
def change_medium(model, new_medium):
    new_model = model.copy()
    new_model.medium = dict(new_medium)
    return new_model
        
# we are making a function named `update_medium` that takes `model` and `new_medium` as arguments. 
# we make a copy of `model` and assign it to `new_model`
# make a variable called `medium` and assign `new_model.medium` to it. 
# `.update()` is a method for dictinaries. we update the `medium` of the new_model with the `new_medium` 
# reassign the the `medium` variable to the new_model.medium (even though we just updated it)

# Questions:
# Why assign the medium attribute to the new_model medium if `medium` was assigned to `new_model` in the second line 
# and it gets updated in the third line? 
# will the `.update()` method work considering that the docs say you have to assign a dictionary to `medium`, you can't just change one thing?
def update_medium(model, new_medium):
    new_model = model.copy()
    medium = new_model.medium
    medium.update(new_medium)
    new_model.medium = medium
    return new_model
    

In [ ]:
# `Medium` is a dictionary and we can tell because the list is stored in {}
# these are other ways of "seeing" variables to find out what they are. 
#print(play_model.medium)
#print(play_model.medium,)
#print(type(play_model.medium))
#print(type(play_model))
#dir(play_model)
#play_model.optimize
#play_model.genes

## Minimal Media 

In [ ]:
from cobra.medium import minimal_medium

play_model2 = load_json_model("/home/gmvaz/2026_GEMs/stm_model_test/STM_v1_0.json")

I am confused here. I thought that `slim_optimize()` does the same thing as `optimize()` but when I tried this code with `optimize()` method. It did the following:

In [ ]:
max_growth_error = play_model2.optimize()

In [ ]:
minimal_medium(play_model2, max_growth_error)

1. I'm not sure what this error means.
2.  I should figure out what the difference is between both methods.

Here's how to get the minimal medium using `slim_optimize()`.

In [ ]:
max_growth = play_model2.slim_optimize()

In [ ]:
minimal_medium(play_model2, max_growth)

To get the minimal media with the smallest amount of active imports

In [ ]:
minimal_medium(play_model2, 0.1, minimize_components=True)

In [ ]:
# minimal_medium(play_model2, 0.8, minimize_components=8, open_exchanges=True)

## Boundary Reactions
Boundary reactions are any reaction that consumes or introduces mass into the system. Exchange, demand, and sink reactions are boundary reactions.  

In [ ]:
play_model2.exchanges[0:5]

In [ ]:
play_model2.demands[0:5]

In [ ]:
play_model2.sinks[0:5]

In [ ]:
play_model2.boundary[0:10]